### Import libaries

In [1]:
import os
import sys
import subprocess
import importlib
sys.path.append('DescriPyTor')

import pandas as pd
import numpy as np
from tqdm import tqdm

from datetime import datetime
from pathlib import Path

from desc_utils import atom_numbering as an
from desc_utils import orca_descriptors as od

ROOT_DIR = os.getcwd()

### Define the source directory containing the orca and xyz files

Provide path to directory or directory of directories containing xyz files path can be 

1. absolute, e.g. "/home/aconner/orca_descriptors/library_pyox_2026_03_03/",

2. relative, e.g. "library_pyox_2026_03_03" (joined to current working directory).

In [2]:
src_dir = "DATA/LNiH2_library_001_to_download_2026_03_24"  # change this as needed

### Define the atoms to be included in the descriptor.

In [3]:
#Define lists for decriptor retrieval 

# 1 -atom descriptors (e.g. Sterimol L, atomic charge)
atoms = ["H1", "H2", "C1", "C2", "N1", "N2", "Ni"]

# 2-atom descriptors (e.g. Sterimol B1, bond length), if order matters for 2-atom descriptors (e.g. Sterimol B1), atoms must be listed in desired order
bonds = [("C1", "N1"), ("N1", "C1"), ("C2", "N2"), ("N2", "C2"), ("Ni", "N1"), ("Ni", "N2"), ("Ni", "H1"), ("Ni", "H2"), ("C1", "C2"), ("C2", "C1")]
pairs_bending = [("H1", "H2"), ("N1", "C2"), ("N2", "C1")] # list of bonds to consider for bend vibration descriptors, if order matters for bend vibration descriptors, atoms must be listed in desired order

# 3-atom descriptors (e.g. Sterimol B5, bond angle), if order matters for 3-atom descriptors (e.g. Sterimol B5), atoms must be listed in desired order
angles = [("H2", "Ni", "H1"), ("Ni", "N1", "C1"), ("Ni", "N2", "C2"), ("N1", "C1", "C2"), ("N2", "C2", "C1"), ("N1", "Ni", "N2"), ("C1", "Ni", "C2")]

# 4-atom descriptors (e.g. dihedral angle), if order matters for 4-atom descriptors (e.g. dihedral angle), atoms must be listed in desired order
dihedrals = [("N1", "C1", "C2", "N2"), ("C1", "C2", "N2", "Ni"), ("C2", "C1", "N1", "Ni")]

# If you want to include quadrant-based descriptors, define the quadrants here. Each quadrant is a list of atom labels. For example, if you want to define one quadrant that includes atoms C1, N1, C2, N2, and Ni, you would define:
#quadrants = [[2, 1, 3, 4, 6]] 


# Atom Numbering

In [4]:
# example for the numbering of the atoms for one molecule:

df = an.compile_final('DATA/LNiH2_library_001_to_download_2026_03_24/AATATIAROAAKIN-SNVBAGLBSA-N/', subdirectory=True)

In [5]:
# Get numbering for all molecules

# Set your ORCA root folder here (contains many conf_* subfolders)
orca_folder = Path(src_dir)

print(f"Current working directory before processing: {os.getcwd()}")

df_num = pd.DataFrame()
for inchi_folder in os.listdir(orca_folder):
    if os.path.isdir(os.path.join(orca_folder, inchi_folder)):
        inchi_folder_path = os.path.join(orca_folder, inchi_folder)
        print(f"Processing {inchi_folder_path} for atom numbering...")
        df = an.compile_final(inchi_folder_path, subdirectory=True)
        df_num = pd.concat([df_num, df], ignore_index=True)

df_num.head()

# add the feather file paths to the df_num dataframe

df_num['feather_path'] = df_num['path'].apply(lambda x: x + x.split('/')[-2] + '.feather')

df_num.head()

Current working directory before processing: /Users/julesschleinitz/Desktop/Code/orca_descriptors
Processing DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/
Processing DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/
Processing DATA/LNiH2_library_001_to_download_2026_03_24/ILENSZIBAAQNCZ-SNVBAGLBSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/ILENSZIBAAQNCZ-SNVBAGLBSA-N/
Processing DATA/LNiH2_library_001_to_download_2026_03_24/YOWGAJZEYGYTFY-SNVBAGLBSA-N for atom numbering...
added slash to end of path, new path is DATA/LNiH2_library_001_to_download_2026_03_24/YOWGAJZEYGYTFY-SNVBAGLBSA-N/
Processing DAT

,xyz_name,path,ligand_type,inchi,SCF_energy,smiles,H1,Ni,H2,N1,C1,C2,N2,ligand_class,feather_path
0,conf_1,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425277,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
1,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425277,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
2,conf_2,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425257,[H]C1=C([H])C([H])=C(F)N[C]1[C]1N[C@@]2([H])c3...,1,2,3,12,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
3,optimization,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425257,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...
4,conf_25,DATA/LNiH2_library_001_to_download_2026_03_24/...,core_5,DVLYVQBEMHJQAQ-SFHVURJKSA-N,-2811.494785,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,6,5,4,pyox,DATA/LNiH2_library_001_to_download_2026_03_24/...


# ORCA Descriptors

### Geometric Descriptors

In [6]:
from desc_utils.data_extraction import *
os.chdir(ROOT_DIR)

results_xyz_dict = get_xyz_descriptors(df_num, atoms, bonds, angles, dihedrals, quadrants=None, overwrite_json=False)
df_xyz_desc = pd.DataFrame.from_dict(results_xyz_dict, orient='index')
df_xyz_desc.set_index('path', inplace=True)
df_xyz_desc.head(5)

  0%|          | 0/606 [00:00<?, ?it/s]

No xyz.json file found in the specified path or overwrite_results is True. Results dictionary will be empty.


  0%|          | 1/606 [00:01<14:24,  1.43s/it]

Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1/xyz.json.
No xyz.json file found in the specified path or overwrite_results is True. Results dictionary will be empty.


  0%|          | 3/606 [00:02<08:58,  1.12it/s]

...

100%|█████████▉| 603/606 [08:13<00:02,  1.10it/s]

Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/KGZUJYDTOJDLJK-SNVBAGLBSA-N/conf_1/xyz.json.
No xyz.json file found in the specified path or overwrite_results is True. Results dictionary will be empty.


100%|██████████| 606/606 [08:14<00:00,  1.22it/s]

Loading existing results from DATA/LNiH2_library_001_to_download_2026_03_24/KGZUJYDTOJDLJK-SNVBAGLBSA-N/conf_2/xyz.json.


,SASA,SASA_Volume,%Vbur_atom_H1_radius_2.0,%Vbur_atom_H1_radius_2.5,%Vbur_atom_H1_radius_3.0,%Vbur_atom_H1_radius_3.5,%Vbur_atom_H1_radius_4.0,%Vbur_atom_H1_radius_4.5,%Vbur_atom_H1_radius_5.0,%Vbur_atom_H2_radius_2.0,...,Angle_H2_Ni_H1,Angle_Ni_N1_C1,Angle_Ni_N2_C2,Angle_N1_C1_C2,Angle_N2_C2_C1,Angle_N1_Ni_N2,Angle_C1_Ni_C2,Dihed_N1_C1_C2_N2,Dihed_C1_C2_N2_Ni,Dihed_C2_C1_N1_Ni
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,463.840531,738.977808,68.217330,58.319768,49.576319,41.425759,35.255803,30.639979,27.199355,68.159220,...,80.285464,112.981331,113.778300,111.707017,119.327437,82.181615,30.439767,-0.264273,-0.930212,1.278973
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,463.840531,738.977808,68.217330,58.319768,49.576319,41.425759,35.255803,30.639979,27.199355,68.159220,...,80.285464,112.981331,113.778300,111.707017,119.327437,82.181615,30.439767,-0.264273,-0.930212,1.278973
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,463.390528,738.279360,68.252841,58.318140,49.604254,41.500885,35.322528,30.694794,27.242715,68.194731,...,80.423225,112.983540,113.755692,111.715757,119.338761,82.195506,30.442198,-0.421224,-0.389638,0.985586
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,463.390528,738.279360,68.252841,58.318140,49.604254,41.500885,35.322528,30.694794,27.242715,68.194731,...,80.423225,112.983540,113.755692,111.715757,119.338761,82.195506,30.442198,-0.421224,-0.389638,0.985586
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,658.628578,1117.755128,67.319861,57.343079,48.592074,41.020430,35.888911,32.580486,30.408021,74.596462,...,79.727473,113.673296,110.255125,112.146020,120.559024,83.229166,30.756306,-0.615986,3.047589,-2.268976


### Charges and Energy Descriptors

In [7]:
results_charges_dict = get_all_charges(df_num, atoms)
df_charges_desc = pd.DataFrame.from_dict(results_charges_dict, orient='index')
df_charges_desc.set_index('path', inplace=True)
df_charges_desc.head(5)

Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW JSON for singlepoint properties...
Parsing GBW J

,mulliken_H1,mulliken_H2,mulliken_C1,mulliken_C2,mulliken_N1,mulliken_N2,mulliken_Ni,loewdin_H1,loewdin_H2,loewdin_C1,...,saniso_N1,saniso_N2,saniso_Ni,siso_H1,siso_H2,siso_C1,siso_C2,siso_N1,siso_N2,siso_Ni
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.077996,-0.097464,0.109601,0.291640,-0.048309,-0.049192,-0.348348,-0.069781,-0.071689,-0.149184,...,366.483239,237.285020,-1920.025043,39.219221,40.217872,37.668341,17.161352,-2.515718,18.126859,-4049.465402
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.077996,-0.097464,0.109601,0.291640,-0.048309,-0.049192,-0.348348,-0.069781,-0.071689,-0.149184,...,366.483239,237.285020,-1920.025043,39.219221,40.217872,37.668341,17.161352,-2.515718,18.126859,-4049.465402
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.079398,-0.097876,0.108511,0.292047,-0.048006,-0.049183,-0.347179,-0.069868,-0.071959,-0.149172,...,367.025926,240.718564,-1934.564548,39.198500,40.214541,37.658881,17.139805,-2.552599,18.228884,-4060.608625
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.079398,-0.097876,0.108511,0.292047,-0.048006,-0.049183,-0.347179,-0.069868,-0.071959,-0.149172,...,367.025926,240.718564,-1934.564548,39.198500,40.214541,37.658881,17.139805,-2.552599,18.228884,-4060.608625
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,-0.098376,-0.074072,0.030477,0.364906,-0.002199,-0.123407,-0.325068,-0.066446,-0.064130,-0.149660,...,397.221237,237.660611,-2606.009812,38.720765,42.552337,36.398284,17.046468,-34.346185,15.348416,-4296.334652


### NBO Descriptors

In [8]:
results_nbo_dict = get_all_nbo_properties(df_num, atoms, basename="singlepoint")
df_nbo_desc = results_nbo_dict.copy()
df_nbo_desc.set_index("path", inplace=True)
df_nbo_desc.head(5)

# filter homo informations to restrict to the range homo-5 to lumo+5
homo_lumo_range = [1, 2, 3, 4, 5]
colums_to_drop = []
for c in df_nbo_desc.columns:
    if re.search(r'_hm\d+_', c):
        if not any(f'_hm{h}_' in c for h in homo_lumo_range):
            colums_to_drop.append(c)
    elif re.search(r'_lp\d+_', c):
        if not any(f'_lp{l}_' in c for l in homo_lumo_range):
            colums_to_drop.append(c)
print(colums_to_drop)

# remove analysis info:
for c in df_nbo_desc.columns:
    if "analysis_info" in c:
        colums_to_drop.append(c)


# Note: the notation hmX means homo - X and lpX means lumo + X.

df_nbo_desc = df_nbo_desc.drop(columns=colums_to_drop)
df_nbo_desc.head(5)


Could not find end of NBO section
Could not find end of NBO section
['nbo_hm78_H1_Ni_type', 'nbo_hm78_H1_Ni_bond_index', 'nbo_hm78_H1_Ni_occupancy', 'nbo_hm78_H1_Ni_energy', 'nbo_hm36_H1_Ni_type', 'nbo_hm36_H1_Ni_bond_index', 'nbo_hm36_H1_Ni_occupancy', 'nbo_hm36_H1_Ni_energy', 'nbo_lp6_H1_type', 'nbo_lp6_H1_bond_index', 'nbo_lp6_H1_occupancy', 'nbo_lp6_H1_energy', 'nbo_lp7_H1_type', 'nbo_lp7_H1_bond_index', 'nbo_lp7_H1_occupancy', 'nbo_lp7_H1_energy', 'nbo_lp8_H1_type', 'nbo_lp8_H1_bond_index', 'nbo_lp8_H1_occupancy', 'nbo_lp8_H1_energy', 'nbo_lp9_H1_type', 'nbo_lp9_H1_bond_index', 'nbo_lp9_H1_occupancy', 'nbo_lp9_H1_energy', 'nbo_lp10_H1_type', 'nbo_lp10_H1_bond_index', 'nbo_lp10_H1_occupancy', 'nbo_lp10_H1_energy', 'nbo_lp41_H2_type', 'nbo_lp41_H2_bond_index', 'nbo_lp41_H2_occupancy', 'nbo_lp41_H2_energy', 'nbo_lp42_H2_type', 'nbo_lp42_H2_bond_index', 'nbo_lp42_H2_occupancy', 'nbo_lp42_H2_energy', 'nbo_lp43_H2_type', 'nbo_lp43_H2_bond_index', 'nbo_lp43_H2_occupancy', 'nbo_lp43_H2_en

,npa_H1_charge,npa_H1_core,npa_H1_valence,npa_H1_rydberg,npa_H1_total,npa_H2_charge,npa_H2_core,npa_H2_valence,npa_H2_rydberg,npa_H2_total,...,nho_directionality_hm_C1_C2_type,nho_directionality_hm_C1_C2_bond_index,nho_directionality_hm_C1_C2_line_theta,nho_directionality_hm_C1_C2_line_phi,nho_directionality_hm_C1_C2_hybrid_theta,nho_directionality_hm_C1_C2_hybrid_phi,nho_directionality_hm_C1_C2_deviation,nho_directionality_hm_C2_N2_hybrid_theta,nho_directionality_hm_C2_N2_hybrid_phi,nho_directionality_hm_C2_N2_deviation
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.21653,0.0,1.21368,0.00285,1.21653,-0.22963,0.0,1.22667,0.00296,1.22963,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.21653,0.0,1.21368,0.00285,1.21653,-0.22963,0.0,1.22667,0.00296,1.22963,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.21662,0.0,1.21378,0.00284,1.21662,-0.23047,0.0,1.22752,0.00295,1.23047,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.21662,0.0,1.21378,0.00284,1.21662,-0.23047,0.0,1.22752,0.00295,1.23047,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,-0.21984,0.0,1.21684,0.00300,1.21984,-0.23576,0.0,1.23294,0.00282,1.23576,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Vertical Redox Descriptors

In [9]:
results_redox_dict = get_redox_descriptors(df_num, atoms)
df_redox_desc = pd.DataFrame.from_dict(results_redox_dict, orient='index')
df_redox_desc.set_index('path', inplace=True)
df_redox_desc.head(5)

,Vertical_Ionization_Potential,Vertical_Electron_Affinity,Hardness,Electronegativity,Electrophilicity,fplus_mulliken_H1,fminus_mulliken_H1,fzero_mulliken_H1,fplus_mulliken_H2,fminus_mulliken_H2,...,fzero_mulliken_C2,fplus_mulliken_N1,fminus_mulliken_N1,fzero_mulliken_N1,fplus_mulliken_N2,fminus_mulliken_N2,fzero_mulliken_N2,fplus_mulliken_Ni,fminus_mulliken_Ni,fzero_mulliken_Ni
path,,,,,,,,,,,,,,,,,,,,,
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.208644,-0.036060,-0.086292,-0.122352,-0.086741,0.052206,0.157895,0.105050,0.071117,0.172430,...,0.041732,0.040563,-0.022216,0.009174,0.058408,-0.021983,0.018212,0.080482,0.263588,0.172035
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1,-0.208644,-0.036060,-0.086292,-0.122352,-0.086741,0.052206,0.157895,0.105050,0.071117,0.172430,...,0.041732,0.040563,-0.022216,0.009174,0.058408,-0.021983,0.018212,0.080482,0.263588,0.172035
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.208691,-0.036117,-0.086287,-0.122404,-0.086819,0.052415,0.158730,0.105573,0.071033,0.172497,...,0.041755,0.040534,-0.022350,0.009092,0.058377,-0.022069,0.018154,0.080286,0.262843,0.171564
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2,-0.208691,-0.036117,-0.086287,-0.122404,-0.086819,0.052415,0.158730,0.105573,0.071033,0.172497,...,0.041755,0.040534,-0.022350,0.009092,0.058377,-0.022069,0.018154,0.080286,0.262843,0.171564
orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/DVLYVQBEMHJQAQ-SFHVURJKSA-N/conf_25,-0.213338,-0.046233,-0.083552,-0.129785,-0.100800,0.050472,0.158524,0.104498,0.059904,0.166209,...,0.042802,0.043624,-0.010402,0.016611,0.065737,-0.028175,0.018781,0.073782,0.241608,0.157695


### Summary of Descriptors

In [10]:
# Build a single summary table matched by molecular path
from desc_utils.data_extraction import _truncate_to_orca_descriptors

# 1) Base table (normalize df_num path so it matches extractor outputs)
_repo_root = Path('desc_utils/data_extraction.py').resolve().parent.parent
df_num_idx = df_num.copy()
df_num_idx['path'] = df_num_idx['path'].map(lambda p: _truncate_to_orca_descriptors(p, _repo_root))
df_num_idx = df_num_idx.set_index("path")

# 2) Concatenate all descriptor tables on index (= path)
total_df = pd.concat(
    [df_num_idx, df_xyz_desc, df_charges_desc, df_redox_desc, df_nbo_desc],
    axis=1,
    join="outer"
)

# 3) Remove duplicated column names (keep first occurrence)
total_df = total_df.loc[:, ~total_df.columns.duplicated()]

# 4) Restore path as a normal column
total_df = total_df.reset_index().rename(columns={"index": "path"})

print("total_df shape:", total_df.shape)
total_df.head(3)

total_df shape: (606, 505)


,path,xyz_name,ligand_type,inchi,SCF_energy,smiles,H1,Ni,H2,N1,...,nho_directionality_hm_C1_C2_type,nho_directionality_hm_C1_C2_bond_index,nho_directionality_hm_C1_C2_line_theta,nho_directionality_hm_C1_C2_line_phi,nho_directionality_hm_C1_C2_hybrid_theta,nho_directionality_hm_C1_C2_hybrid_phi,nho_directionality_hm_C1_C2_deviation,nho_directionality_hm_C2_N2_hybrid_theta,nho_directionality_hm_C2_N2_hybrid_phi,nho_directionality_hm_C2_N2_deviation
0,orca_descriptors/DATA/LNiH2_library_001_to_dow...,conf_1,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425277,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,orca_descriptors/DATA/LNiH2_library_001_to_dow...,optimization,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425277,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,1,2,3,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,orca_descriptors/DATA/LNiH2_library_001_to_dow...,conf_2,core_5,IJHJNYMOYGPPGZ-OCCSQVGLSA-N,-2371.425257,[H]C1=C([H])C([H])=C(F)N[C]1[C]1N[C@@]2([H])c3...,1,2,3,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# DescriPyTor

### Convert the ORCA output files to pseudo Gaussian log files for compatibility with descripytor generation. 

This step is necessary because descripy is designed to work with Gaussian log files, and the ORCA output files need to be converted to this format for descripy to process them correctly.

In [11]:
# convert orca files to log files | a log file is created for each conf_* subfolder in the orca_folder, and contains the same information as the orca output file but in a format that can be read by the descriptor calculation code
script_path = Path("orca2gaussian/orca_2_gauss.py")

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path.resolve()}")
if not orca_folder.exists():
    raise FileNotFoundError(f"Folder not found: {orca_folder.resolve()}")


try:
    subprocess.run(
                [sys.executable, str(script_path), "--orca-folder", str(orca_folder)],
                capture_output=True,
                text=True,
                check=True,
            )

except subprocess.CalledProcessError as exc:
    print(f"Failed for {orca_folder}: {exc.stderr.strip() or exc}")

### DescriPyTor

#### Convert the log files to feather files for use in DescriPyTor.

In [ ]:
# Load Descripytor feather file utils
DESCRIPYTOR_DIR = Path("DescriPyTor/MolFeatures")
OVERWRITE_FEATHER = False  # Set to True to overwrite existing feather files, False to skip if feather file already exists
print(f"Setting DESCRIPYTOR_DIR to: {DESCRIPYTOR_DIR}")
sys.path.append(DESCRIPYTOR_DIR)
sys.path.append(os.path.join(DESCRIPYTOR_DIR, 'M3_modeler'))
sys.path.append(os.path.join(DESCRIPYTOR_DIR, 'M2_data_extractor'))
sys.path.append(os.path.join(DESCRIPYTOR_DIR, 'utils'))

try:
    from data_extractor import Molecules, extract_connectivity
    from feather_extractor import logs_to_feather
    import gaussian_handler
    from gaussian_handler import feather_file_handler
    from help_functions import dict_to_horizontal_df, get_df_from_file, pick_samples_to_remove_for_distribution
    import help_functions
    import visualize

    print("All modules imported successfully.")
    
except ModuleNotFoundError as e:
    print(f"Module import failed: {e} \nPlease ensure all required modules are installed and available in the path specified on the right hand corner of the notebook.")


pd.set_option('display.max_columns', None)

# log_dir = os.path.join(examples_path, 'logs_example')

# iterate over all the log files in the orca_folder and check if they exist, print the path of the log file if it exists, otherwise print a warning message, and extract the log files to a feather file.
pwd_before = os.getcwd()
print(f"Current working directory: {os.getcwd()}")
for inchi_folder in os.listdir(orca_folder):
    os.chdir(os.path.join(orca_folder, inchi_folder))
    for conf_folder in os.listdir('.'):
        if conf_folder.startswith('conf_') and os.path.isdir(conf_folder):
            os.chdir(conf_folder)
            log_file_path = None
            for f in os.listdir('.'):
                if f.endswith('.log'):
                    log_file_path = os.path.join(os.getcwd(), f)
            if log_file_path is None:
                print(f"No log file found in {os.getcwd()}")
            else:
                feather_file = log_file_path.replace('.log', '.feather')
                if os.path.exists(feather_file) and not OVERWRITE_FEATHER:
                    print(f"Feather file already exists, skipping: {feather_file}")
                else:
                    try:
                        logs_to_feather(os.getcwd(), feather_dir=os.getcwd())
                        print(f"Logs successfully converted to feather format at: {os.getcwd()}")
                    except Exception as e:
                        print(f"Error converting logs in {os.getcwd()} to feather format: {e}")
            os.chdir('..')
        
    os.chdir(pwd_before)
    

Setting DESCRIPYTOR_DIR to: DescriPyTor/MolFeatures
Module import failed: No module named 'modeling' 
Please ensure all required modules are installed and available in the path specified on the right hand corner of the notebook.
Current working directory: /Users/julesschleinitz/Desktop/Code/orca_descriptors
conf_1.log: NBO, Hirshfeld and CM5 charges
Data saved to conf_1.feather
Done!
Logs successfully converted to feather format at: /Users/julesschleinitz/Desktop/Code/orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_1
conf_2.log: NBO, Hirshfeld and CM5 charges
Data saved to conf_2.feather
Done!
Logs successfully converted to feather format at: /Users/julesschleinitz/Desktop/Code/orca_descriptors/DATA/LNiH2_library_001_to_download_2026_03_24/IJHJNYMOYGPPGZ-OCCSQVGLSA-N/conf_2
conf_25.log: NBO, Hirshfeld and CM5 charges
Data saved to conf_25.feather
Done!
Logs successfully converted to feather format at: /Users/julesschleinitz/Desktop/Code/o

#### Extract DescriPyTor descriptors (vibrations, dipoles and sterimol descriptors)

In [ ]:

os.chdir(ROOT_DIR)

# renumbering of the atoms in the feather files according to the df_num numberings
renumbering_convention = {"H1": 1, "H2": 2, "Ni": 3, "N1": 4, "C1": 5, "C2": 6, "N2": 7}

desc_all = {}
for i, row in tqdm(df_num.iterrows(), total=df_num.shape[0]):
    os.chdir(ROOT_DIR)
    desc_mol = {}
    # read path
    feather_dir = row['path']
    desc_mol['feather_dir'] = feather_dir
    # create Molecule object and renumber atoms according to the df_num numberings
    mols = Molecules(molecules_dir_name=feather_dir)

    # global descriptors
    ## 1-atom descriptors
    charges_dict    = mols.get_charge_df_dict(atom_indices=[row[x] for x in atoms]) # NBO
    index = charges_dict.index[0]
    for j, atom in enumerate(atoms):
        desc_mol[f'charge_NBO_{atom}'] = charges_dict.loc[index, f"nbo_atom_{row[atom]}"]

    ## 2-atom descriptors
    # sterimol descriptors (uses "bonds" list defined above to determine which sterimol descriptors to calculate)
    sterimol_descriptors    = mols.get_sterimol_dict(atom_indices=[[row[x], row[y]] for x, y in bonds])
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'B1_{x1}-{x2}']           = sterimol_descriptors.loc[index, f'B1_{row[x1]}-{row[x2]}']
        desc_mol[f'B5_{x1}-{x2}']           = sterimol_descriptors.loc[index, f'B5_{row[x1]}-{row[x2]}']
        desc_mol[f'L_{x1}-{x2}']            = sterimol_descriptors.loc[index, f'L_{row[x1]}-{row[x2]}']
        desc_mol[f'loc_B5_{x1}-{x2}']       = sterimol_descriptors.loc[index, f'loc_B5_{row[x1]}-{row[x2]}']
        desc_mol[f'B1-B5_angle_{x1}-{x2}']  = sterimol_descriptors.loc[index, f'B1_B5_angle_{row[x1]}-{row[x2]}']

    bond_length_dict        = mols.get_bond_length_dict(atom_pairs=[[row[x], row[y]] for x, y in bonds]) # dataframe with bond length descriptors for specified bonds
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'dist_{x1}-{x2}'] = bond_length_dict.loc[index, f'0_bond_length_{row[x1]}-{row[x2]}']

    vibrations_dict         = mols.get_stretch_vibration_dict(atom_pairs=[[row[x], row[y]] for x, y in bonds]) # dataframe with vibration descriptors for all molecules in the feather file, can be filtered for specific vibrations of interest based on the atoms involved in the vibration
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'vib_freq_{x1}-{x2}'] = vibrations_dict.loc[index, f'Frequency_Stretch_{row[x1]}_{row[x2]}']
        desc_mol[f'vib_amp_{x1}-{x2}'] = vibrations_dict.loc[index, f'Amplitude_Stretch_{row[x1]}_{row[x2]}']

    vibrations_bend_dict    = mols.get_bend_vibration_dict(atom_pairs=[[row[x], row[y]] for x, y in pairs_bending]) # dataframe with bend vibration descriptors for specified angles
    for j, (x1, x2) in enumerate(pairs_bending):
        desc_mol[f'vib_freq_bend_{x1}-{x2}'] = vibrations_bend_dict.loc[index, f'Frequency_Bending_{row[x1]}-{row[x2]}']
        desc_mol[f'vib_amp_bend_{x1}-{x2}']  = vibrations_bend_dict.loc[index, f'Cross_mag_Bending_{row[x1]}-{row[x2]}']
    
    charges_diff_dict       = mols.get_charge_diff_df_dict(atom_indices=[[row[x], row[y]] for x, y in bonds]) # dataframe with charge difference descriptors for specified bonds
    for j, (x1, x2) in enumerate(bonds):
        desc_mol[f'charge_NBO_diff_{x1}-{x2}'] = charges_diff_dict.loc[index, f'nbo_diff_{row[x1]}-{row[x2]}']

    ## 3-atom descriptors
    dipole_ni = mols.get_dipole_dict(atom_indices=[row["Ni"], row["N2"], row["N1"]]) # dataframe with dipole moment descriptors for specified atom
    desc_mol['dipole'] = dipole_ni.loc[index, f'total_dipole_{row["Ni"]}-{row["N2"]}-{row["N1"]}']
    desc_mol['dipole_x_Ni-N2-N1'] = dipole_ni.loc[index, f'dipole_x_{row["Ni"]}-{row["N2"]}-{row["N1"]}']
    desc_mol['dipole_y_Ni-N2-N1'] = dipole_ni.loc[index, f'dipole_y_{row["Ni"]}-{row["N2"]}-{row["N1"]}']
    desc_mol['dipole_z_Ni-N2-N1'] = dipole_ni.loc[index, f'dipole_z_{row["Ni"]}-{row["N2"]}-{row["N1"]}']

    angles_dict = mols.get_bond_angle_dict(atom_indices=[[row[x], row[y], row[z]] for x, y, z in angles])
    for j, (x1, x2, x3) in enumerate(angles):
        desc_mol[f'angle_{x1}-{x2}-{x3}'] = angles_dict.loc[index, f'0_angle_[{row[x1]}, {row[x2]}, {row[x3]}]']

    desc_all[i] = desc_mol

df_desc = pd.DataFrame.from_dict(desc_all, orient='index')
df_desc.head(5)

  0%|          | 0/606 [00:00<?, ?it/s]

Molecules Loaded: ['conf_1.feather'] Failed Molecules: []


  0%|          | 1/606 [00:00<08:22,  1.20it/s]

Calculating stretch vibration for atoms [[6, 12], [12, 6], [5, 4], [4, 5], [2, 12], [2, 4], [2, 1], [2, 3], [6, 5], [5, 6]] with threshold 1400 
 Remember : ALWAYS LOOK AT THE RESULTING VIBRATION
Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_1 for [2, 12] 

Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_1 for [2, 4] 

Molecules Loaded: ['conf_1.feather'] Failed Molecules: []


  0%|          | 2/606 [00:01<08:31,  1.18it/s]

...

 36%|███▌      | 218/606 [06:37<07:21,  1.14s/it]

Calculating stretch vibration for atoms [[6, 13], [13, 6], [5, 4], [4, 5], [2, 13], [2, 4], [2, 1], [2, 3], [6, 5], [5, 6]] with threshold 1400 
 Remember : ALWAYS LOOK AT THE RESULTING VIBRATION
Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_7 for [2, 13] 

Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_7 for [2, 4] 

Molecules Loaded: ['conf_5.feather'] Failed Molecules: []


 36%|███▌      | 219/606 [06:38<07:20,  1.14s/it]

Calculating stretch vibration for atoms [[6, 13], [13, 6], [5, 4], [4, 5], [2, 13], [2, 4], [2, 1], [2, 3], [6, 5], [5, 6]] with threshold 1400 
 Remember : ALWAYS LOOK AT THE RESULTING VIBRATION
Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_5 for [2, 13] 

Strech Vibration Error: the following bonds do not exist-check atom numbering in molecule: 
 conf_5 for [2, 4] 

Molecules Loaded: ['conf_5.feather'] Failed Molecules: []


 36%|███▋      | 220/606 [06:39<07:22,  1.15s/it]

# Combining ORCA and DescriPyTor descriptors

In [ ]:
import re

# --- 1) Build merge key (conf folder) for total_df ---
total_work = total_df.drop(columns=atoms).copy()

def _normalize_conf_dir(p):
    if pd.isna(p):
        return np.nan
    p = str(p).replace("\\", "/")
    if p.endswith(".feather"):
        p = os.path.dirname(p)
    p = p.rstrip("/")
    p = re.sub(r"^orca_descriptors/", "", p)  # align with df_desc paths
    return p

total_work["conf_dir"] = total_work["path"].map(_normalize_conf_dir)

# --- 2) Build merge key for df_desc ---
desc_work = df_desc.copy()
if "feather_dir" not in desc_work.columns:
    raise KeyError("df_desc must contain 'feather_dir'.")

desc_work["conf_dir"] = desc_work["feather_dir"].map(_normalize_conf_dir)

# --- 3) Combine descriptor tables ---
combined_df = total_work.merge(
    desc_work,
    on="conf_dir",
    how="outer",
    suffixes=("_total", "_desc")
)

# If duplicate energy columns exist, keep one unified SCF_energy
if "SCF_energy" not in combined_df.columns:
    scf_candidates = [c for c in combined_df.columns if c.startswith("SCF_energy")]
    if scf_candidates:
        combined_df["SCF_energy"] = combined_df[scf_candidates].bfill(axis=1).iloc[:, 0]

# --- 4) Group key = inchi folder (path without /conf_XX) ---
combined_df["inchi_folder"] = combined_df["conf_dir"].str.replace(
    r"/conf_\d+$", "", regex=True
)

# --- 5) Boltzmann aggregation per inchi folder ---
T = 298.15
kB_hartree_per_K = 3.166811563e-6  # Hartree/K

if "SCF_energy" not in combined_df.columns:
    raise KeyError("SCF_energy column not found after merge.")

numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()

rows = []
for inchi_folder, g in combined_df.groupby("inchi_folder", dropna=True):
    g = g.copy()
    e = pd.to_numeric(g["SCF_energy"], errors="coerce")
    if e.notna().sum() == 0:
        continue

    e_min = e.min()
    idx_low = e.idxmin()

    dE = e - e_min
    w = np.exp(-(dE / (kB_hartree_per_K * T)))
    w = w / np.nansum(w)

    out = {
        "inchi_folder": inchi_folder,
        "inchi": os.path.basename(inchi_folder),
        "min_energy_value": e_min,
        "smiles": g["smiles"].iloc[0] if "smiles" in g.columns else np.nan,
        "ligand_class": g["ligand_class"].iloc[0] if "ligand_class" in g.columns else np.nan
    }

    for col in numeric_cols:
        v = pd.to_numeric(g[col], errors="coerce")

        out[f"{col}_min"] = v.min()
        out[f"{col}_max"] = v.max()
        out[f"{col}_lowE"] = g.loc[idx_low, col] if idx_low in g.index else np.nan

        valid = v.notna() & w.notna()
        out[f"{col}_boltz"] = (
            np.average(v[valid], weights=w[valid]) if valid.any() else np.nan
        )

    rows.append(out)

df_boltzmann_summary = pd.DataFrame(rows).sort_values("inchi_folder").reset_index(drop=True)
df_boltzmann_summary.drop(columns=["min_energy_value"], inplace=True)

print("combined_df shape:", combined_df.shape)
print("df_boltzmann_summary shape:", df_boltzmann_summary.shape)
df_boltzmann_summary.head()

### Remove noisy descriptors

In [ ]:
import importlib
import desc_utils.data_extraction as de

importlib.reload(de)

from desc_utils.data_extraction import filter_numeric_descriptors

df_num_desc_filtered, report = filter_numeric_descriptors(
    df_boltzmann_summary,
    remove_low_occupancy_nbo=True,
    occupancy_boltz_threshold=0.001,
    corr_threshold=0.95,
    corr_pruning_mode="bridge_star_centers", #"component_ranked",
    ratio_threshold=0.01,
    orbital_filter="remove rydbergs",
    cor_orbital_filter="remove cor",
    ni_orbital_filter="remove 4f rydbergs",    # new filter
    nmr_filter="remove aniso",
    bond_analysis_filter="remove phi theta",
)

print("Initial numeric descriptors:", report["initial_numeric_count"])
print("Removed (NBO low occupancy orbitals):", len(report["removed_low_occ_nbo_orbitals"]))
print("Removed (NBO low occupancy columns):", len(report["removed_low_occ_nbo_columns"]))
print("Removed (Rydberg, except Ni):", len(report["removed_rydberg"]))
print("Removed (Ni 4f Rydbergs):", len(report["removed_ni_4f_rydberg"]))
print("Removed (NMR aniso):", len(report["removed_nmr_aniso"]))
print("Removed (phi/theta bond analysis):", len(report["removed_phi_theta"]))
print("Removed (zero std):", len(report["removed_zero_std"]))
print("Removed (std/abs(mean) < 1%):", len(report["removed_low_ratio"]))
print("Removed (high correlation):", len(report["removed_high_corr"]))
print("Final numeric descriptors:", report["final_numeric_count"])

# filter_numeric_descriptors returns only numeric columns, so re-attach meta columns from df_boltzmann_summary
meta_cols = ["smiles", "inchi", "inchi_folder", "ligand_class"]
for col in meta_cols:
    if col in df_boltzmann_summary.columns:
        df_num_desc_filtered[col] = df_boltzmann_summary[col].values

df_num_desc_filtered.head()


In [ ]:
# save to csv
from datetime import datetime
output_csv_path = datetime.now().strftime("%Y%m%d") + "_" + src_dir.split('/')[-1] + "_descriptors_" + ".csv"
df_num_desc_filtered.to_csv(output_csv_path, index=False)